In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mysql.connector
import random
import time
from datetime import datetime, timedelta

# Conexión con MySQL (northwind_mysql = tu base fuente)
conexion = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="miclave",
    database="northwind_mysql"
)

cursor = conexion.cursor()

# Reutiliza customerID, employeeID y shipVia que YA existen en la tabla,
# ya que no hay llaves foráneas activas pero sí queremos datos coherentes.
cursor.execute("SELECT DISTINCT customerID FROM Orders WHERE customerID IS NOT NULL")
customer_ids = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT DISTINCT employeeID FROM Orders WHERE employeeID IS NOT NULL")
employee_ids = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT DISTINCT shipVia FROM Orders WHERE shipVia IS NOT NULL")
ship_via_ids = [row[0] for row in cursor.fetchall()]

# Calcula el siguiente orderID (la tabla no es autoincremental)
cursor.execute("SELECT MAX(orderID) FROM Orders")
siguiente_id = (cursor.fetchone()[0] or 0) + 1

ciudades = [
    ("Quito", "Pichincha", "170101", "Ecuador"),
    ("Guayaquil", "Guayas", "090101", "Ecuador"),
    ("Cuenca", "Azuay", "010101", "Ecuador"),
    ("Bogotá", "Cundinamarca", "110111", "Colombia"),
    ("Lima", "Lima", "150101", "Peru"),
]

print("======================================")
print(" Generador de pedidos (Orders)")
print("======================================")
print("Insertando un pedido nuevo cada 10 segundos")
print("Presiona Ctrl+C (o el botón de stop del kernel) para detenerlo.\n")

try:
    while True:
        order_date = datetime.now()
        required_date = order_date + timedelta(days=random.randint(5, 20))
        # Deja algunos pedidos sin enviar aún (shippedDate NULL), como en la vida real
        shipped_date = (
            order_date + timedelta(days=random.randint(1, 4))
            if random.random() > 0.3 else None
        )
        ciudad, region, cp, pais = random.choice(ciudades)

        sql = """
        INSERT INTO Orders
        (orderID, customerID, employeeID, orderDate, requiredDate, shippedDate,
         shipVia, freight, shipName, shipAddress, shipCity, shipRegion,
         shipPostalCode, shipCountry)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        valores = (
            siguiente_id,
            random.choice(customer_ids),
            random.choice(employee_ids),
            order_date,
            required_date,
            shipped_date,
            random.choice(ship_via_ids),
            round(random.uniform(5, 500), 2),
            f"Pedido automatico {siguiente_id}",
            "Av. Simulada 123",
            ciudad,
            region,
            cp,
            pais,
        )

        cursor.execute(sql, valores)
        conexion.commit()

        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"orderID: {siguiente_id} | cliente: {valores[1]} | "
            f"empleado: {valores[2]} | ciudad: {ciudad} | freight: ${valores[7]:.2f}"
        )

        siguiente_id += 1
        time.sleep(10)

except KeyboardInterrupt:
    print("\nProceso detenido.")

finally:
    cursor.close()
    conexion.close()
    print("Conexión MySQL cerrada.")


 Generador de pedidos (Orders)
Insertando un pedido nuevo cada 10 segundos
Presiona Ctrl+C (o el botón de stop del kernel) para detenerlo.

[14:46:06] orderID: 11078 | cliente: TRADH | empleado: 3 | ciudad: Lima | freight: $10.26
[14:46:16] orderID: 11079 | cliente: BERGS | empleado: 6 | ciudad: Cuenca | freight: $414.79
[14:46:26] orderID: 11080 | cliente: HILAA | empleado: 8 | ciudad: Cuenca | freight: $24.89
[14:46:36] orderID: 11081 | cliente: LONEP | empleado: 4 | ciudad: Bogotá | freight: $243.23
[14:46:46] orderID: 11082 | cliente: CACTU | empleado: 7 | ciudad: Lima | freight: $147.59
[14:46:57] orderID: 11083 | cliente: QUEDE | empleado: 4 | ciudad: Lima | freight: $151.31
[14:47:07] orderID: 11084 | cliente: OTTIK | empleado: 3 | ciudad: Cuenca | freight: $456.69

Proceso detenido.
Conexión MySQL cerrada.
